In [1]:
from utils.loaders.load_solar import load_solar_profile
from utils.loaders.load_re_costs import load_re_costs
import os
from dotenv import load_dotenv
# import pandas as pd
# import numpy as np
# import boto3
# from io import BytesIO

In [2]:
df, solar = load_solar_profile(
    site_id=110, 
    year=2023,
    bucket_name="annaiecc",
    folder_prefix="solar_data_parquet_v3"
)


In [3]:
df.head(n=24)

,source_name,timestamp,P_kWperkWp,year
0,Edge Moor,2023-01-01 00:00:00,0.00000,2023
1,Edge Moor,2023-01-01 01:00:00,0.00000,2023
2,Edge Moor,2023-01-01 02:00:00,0.00000,2023
3,Edge Moor,2023-01-01 03:00:00,0.00000,2023
4,Edge Moor,2023-01-01 04:00:00,0.00000,2023
5,Edge Moor,2023-01-01 05:00:00,0.00000,2023
6,Edge Moor,2023-01-01 06:00:00,0.00000,2023
7,Edge Moor,2023-01-01 07:00:00,0.00018,2023
8,Edge Moor,2023-01-01 08:00:00,0.12237,2023
9,Edge Moor,2023-01-01 09:00:00,0.34081,2023


In [4]:
solar

array([0., 0., 0., ..., 0., 0., 0.], shape=(8760,))

### Load RE costs

In [5]:
re_costs = load_re_costs(
    '../../BNEF/output/bnef_solar_pv_costs_v4.csv',
    '../../BNEF/output/battery_2025_cost.csv',
    'USA',
    '2025',
)

In [6]:
re_costs

{'financial': {'CRF': 0.083846},
 'solar': {'Capex': 974.82, 'Fixed O&M': 15400.800000000001},
 'battery': {'Pack': 106.43758666666666,
  'Rack': 30.703149999999997,
  'BOS + EMS': 35.820341666666664,
  'EPC': 53.21879333333333,
  'PCS + Overhead': 100.29695666666666,
  'Fixed O&M': 10456.2}}

### Load Demand

In [8]:
from utils.loaders.load_demand import get_peak_hours_binary

In [9]:
gas_allowed, aligned_demand = get_peak_hours_binary(110, df, peak_cutoff=0, folder="inputs/demand", start_year=2016, end_year=2024, flat_block=True, flat_block_load_mw=200)

Flat block mode: Gas allowed all 8760 hours
Flat demand profile: 200 MW for all hours


In [10]:
aligned_demand

array([200, 200, 200, ..., 200, 200, 200])

### Load Gas

In [15]:
import duckdb
import boto3
import pandas as pd

def load_site_params(site_id, bucket_name, folder_prefix="climate_trace_parquet"):
    s3_prefix = f"s3://{bucket_name}/{folder_prefix}/facility_master.parquet"

    # Grab credentials from boto3 session (works with IAM role, env vars, ~/.aws/credentials)
    session = boto3.Session()
    creds = session.get_credentials().get_frozen_credentials()
    region = session.region_name or 'us-east-1'

    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"""
        SET s3_region='{region}';
        SET s3_access_key_id='{creds.access_key}';
        SET s3_secret_access_key='{creds.secret_key}';
        SET s3_session_token='{creds.token or ''}';
    """)

    query = f"""
        SELECT *
        FROM read_parquet('{s3_prefix}')
        WHERE source_id = '{site_id}'
    """

    df = con.execute(query).df()
    return df

In [16]:
site_df = load_site_params(110, 'annaiecc', folder_prefix="climate_trace_parquet")

HTTPException: HTTP Error: Unable to connect to URL "https://annaiecc.s3.us-west-2.amazonaws.com/climate_trace_parquet/facility_master.parquet": 301 (Moved Permanently).

Bad Request - this can be caused by the S3 region being set incorrectly.
* Provided region is: "us-west-2"
* Correct region is: "us-east-1"

LINE 3:         FROM read_parquet('s3://annaiecc/climate_trace_parquet/facility_...
                     ^

In [28]:
site_df

,source_id,source_name,source_type,iso3_country,sector,subsector,start_time,end_time,lat,lon,...,modified_date,lat_lon,reporting_entity,sector_id,native_source_id,emission_type,source_version,emission,sector_1,emission_type_1
0,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2021-01-01 00:00:00,2021-01-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2,v5.1.0,co2,power,co2
1,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2021-02-01 00:00:00,2021-02-28 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2,v5.1.0,co2,power,co2
2,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2021-03-01 00:00:00,2021-03-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2,v5.1.0,co2,power,co2
3,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2021-04-01 00:00:00,2021-04-30 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2,v5.1.0,co2,power,co2
4,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2021-05-01 00:00:00,2021-05-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2,v5.1.0,co2,power,co2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2025-05-01 00:00:00,2025-05-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2e_100yr,v5.1.0,co2e_100yr,power,co2e_100yr
677,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2025-06-01 00:00:00,2025-06-30 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2e_100yr,v5.1.0,co2e_100yr,power,co2e_100yr
678,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2025-07-01 00:00:00,2025-07-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2e_100yr,v5.1.0,co2e_100yr,power,co2e_100yr
679,110,Edge Moor,"gas, oil",USA,power,electricity-generation,2025-08-01 00:00:00,2025-08-31 00:00:00,39.7412,-75.5055,...,2025-10-13 23:03:27,NaN,climate-trace,61.0,TRRRCZLCIE,co2e_100yr,v5.1.0,co2e_100yr,power,co2e_100yr
